Carregamento dos Dados da Rede

In [14]:
import py_dss_interface
from py_dss_toolkit import dss_tools
import pandas as pd
import pickle

dss = py_dss_interface.DSS()

dss_file = r"C:\Dados_teste\OpenDSS\CBA\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"
#dss_file = r"C:\Users\IVAN SANTOS\Documents\OpenDSS\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"

dss_tools.update_dss(dss)

dss.text(f"compile [{dss_file}]")
dss.text("Redirect 'PV_System_120_MeiaPonte.dss'")

with open("dataframes.pkl", "rb") as f:
    dfs_monitor_pv = pickle.load(f)

loads_df = dss_tools.model.loads_df

#Limpeza dos dados

floats = ["kv", "kw", "vmaxpu", "vminpu", "pf"]
strings = ["name", "daily", "conn"]

loads_df[floats] = loads_df[floats].astype(float)
loads_df[strings] = loads_df[strings].astype(str)
loads_df["phases"] = loads_df["phases"].astype(int)

loadshape_p = [0.63431595, 0.5781472, 0.53907255, 0.53100273, 0.525749, 0.5497442, 0.58774194, 0.57231744, 0.60902958, 0.66933898, 0.68768531, 0.7447487, 0.79201143, 0.74954746, 0.76405228, 0.75331663, 0.77208036, 0.81570054, 0.97013191, 1.0, 0.97971692, 0.92489638, 0.8190189, 0.74192983]
loadshape_q = [0.57151726, 0.51464415, 0.47607553, 0.46821595, 0.46296368, 0.48604702, 0.5237456, 0.50838673, 0.54647068, 0.61033922, 0.63035948, 0.69150093, 0.74493019, 0.6987781, 0.71553876, 0.703765, 0.72365629, 0.77090208, 0.9596657, 1.0, 0.97331006, 0.90212777, 0.77270589, 0.68527089]

loads_df = loads_df.sort_values(by="kw", ascending=False).reset_index(drop=True)

display(loads_df[["name", "phases", "kv", "kw", "daily"]])

print(dfs_monitor_pv.keys())
#display(dfs_monitor_pv["pv_27_120_gd_df"])
print(dfs_monitor_pv["pv_27_120_gd_df"].columns)

,name,phases,kv,kw,daily
0,m1_trf_5286964a,3,0.38,69.143341,CURVA
1,m2_trf_5286964a,3,0.38,69.143341,CURVA
2,m2_trf_5286812a,3,0.38,62.255034,CURVA
3,m1_trf_5286812a,3,0.38,62.255034,CURVA
4,m1_trf_5552819a,3,0.38,51.943260,CURVA
...,...,...,...,...,...
301,mt_10020770241_nt_m2,3,13.80,0.322847,CURVA
302,m2_trf_5286699a,3,0.38,0.268829,CURVA
303,m1_trf_5286699a,3,0.38,0.268829,CURVA
304,m2_trf_5702287a,3,0.38,0.232467,CURVA


dict_keys(['pv_1_60_gd_df', 'pv_2_60_gd_df', 'pv_3_60_gd_df', 'pv_4_60_gd_df', 'pv_5_60_gd_df', 'pv_6_60_gd_df', 'pv_7_60_gd_df', 'pv_8_60_gd_df', 'pv_9_60_gd_df', 'pv_10_60_gd_df', 'pv_11_60_gd_df', 'pv_12_60_gd_df', 'pv_13_60_gd_df', 'pv_14_60_gd_df', 'pv_15_60_gd_df', 'pv_16_60_gd_df', 'pv_17_60_gd_df', 'pv_18_60_gd_df', 'pv_19_60_gd_df', 'pv_20_60_gd_df', 'pv_21_60_gd_df', 'pv_22_60_gd_df', 'pv_23_60_gd_df', 'pv_24_60_gd_df', 'pv_25_60_gd_df', 'pv_26_60_gd_df', 'pv_27_60_gd_df', 'pv_28_60_gd_df', 'pv_29_60_gd_df', 'pv_30_60_gd_df', 'pv_31_60_gd_df', 'pv_32_60_gd_df', 'pv_33_60_gd_df', 'pv_34_60_gd_df', 'pv_35_60_gd_df', 'pv_36_60_gd_df', 'pv_37_60_gd_df', 'pv_38_60_gd_df', 'pv_39_60_gd_df', 'pv_40_60_gd_df', 'pv_41_60_gd_df', 'pv_42_60_gd_df', 'pv_43_60_gd_df', 'pv_44_60_gd_df', 'pv_45_60_gd_df', 'pv_46_60_gd_df', 'pv_47_60_gd_df', 'pv_48_60_gd_df', 'pv_49_60_gd_df', 'pv_50_60_gd_df', 'pv_51_60_gd_df', 'pv_52_60_gd_df', 'pv_53_60_gd_df', 'pv_54_60_gd_df', 'pv_55_60_gd_df', 'pv_56_6

Criação do DataFrame com a curva das irradiâncias

In [19]:
import numpy as np

loadshape_dict = {}

for nome, kw in zip(loads_df["name"], loads_df["kw"]):

    loadshape_dict[nome] = kw * np.array(loadshape_p)

loadshape_df = pd.DataFrame(loadshape_dict)


storage_dict = {}

i = 0

for chave, df in dfs_monitor_pv.items():

    if "120_gd_df" in chave:

        if " P3 (kW)" not in df.columns:

            df["Total Power"] = (
                df[" P1 (kW)"] +
                df[" P2 (kW)"]
            )

        else:

            df["Total Power"] = (
                df[" P1 (kW)"] +
                df[" P2 (kW)"] +
                df[" P3 (kW)"]
            )

        df = df.head(24)

        storage_kw = 2 * loads_df.loc[i, "kw"]

        curva = (df["Total Power"].reset_index(drop=True) + loadshape_df.iloc[:, i]) / storage_kw

        saldo = curva.sum()

        if abs(saldo) > 0.05:

            curva.iloc[-1] -= saldo

        i += 1

        storage_dict[f"Battery_{i}"] = curva


storage_df = pd.DataFrame(storage_dict)

display(storage_df)



,Battery_1,Battery_2,Battery_3,Battery_4,Battery_5,Battery_6,Battery_7,Battery_8,Battery_9,Battery_10,...,Battery_111,Battery_112,Battery_113,Battery_114,Battery_115,Battery_116,Battery_117,Battery_118,Battery_119,Battery_120
0,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,...,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158,0.317158
1,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,...,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074,0.289074
2,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,...,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536,0.269536
3,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,...,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501,0.265501
4,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,...,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875,0.262875
5,0.224496,0.224496,0.224537,0.224537,0.224598,0.224598,0.224607,0.224607,0.224608,0.224608,...,0.224747,0.224747,0.224748,0.224748,0.224749,0.224749,0.224750,0.224750,0.224750,0.224750
6,0.005283,0.005283,0.006532,0.006532,0.008402,0.008402,0.008662,0.008662,0.008701,0.008701,...,0.012948,0.012948,0.012986,0.012986,0.013019,0.013019,0.013036,0.013036,0.013037,0.013037
7,-0.397468,-0.397468,-0.392977,-0.392977,-0.383492,-0.383492,-0.382169,-0.382169,-0.381971,-0.381971,...,-0.360423,-0.360423,-0.360229,-0.360229,-0.360064,-0.360064,-0.359977,-0.359977,-0.359971,-0.359971
8,-0.678738,-0.678738,-0.673950,-0.673950,-0.666748,-0.666748,-0.665770,-0.665770,-0.665616,-0.665616,...,-0.626662,-0.626662,-0.626275,-0.626275,-0.625940,-0.625940,-0.625774,-0.625774,-0.625757,-0.625757
9,-0.852050,-0.852050,-0.845222,-0.845222,-0.835005,-0.835005,-0.833580,-0.833580,-0.833367,-0.833367,...,-0.790285,-0.790285,-0.789727,-0.789727,-0.789252,-0.789252,-0.789004,-0.789004,-0.788988,-0.788988


Escrita do Código DSS

In [23]:
#30 Baterias
def code_writer(number, f):
    linhas = []
    for i in range(number):
        shape = f"New LoadShape.StorageCurve_{i + 1} npts=24 interval=1\n"

        bateria = storage_df[f"Battery_{i + 1}"].round(2)
        
        curve = "~ mult=[" + ", ".join(map(str, bateria)) + "]\n"
        phase = loads_df.loc[i, "phases"]
        kv = loads_df.loc[i, "kv"]
        bus = loads_df.loc[i, "bus1"]
        kwrated = 2*loads_df.loc[i, "kw"]
        kwhrated = 16*loads_df.loc[i, "kw"]
        stored = "%stored=50"

        battery = f"New Storage.Battery_{i+1} phases={phase} bus1={bus} kv={kv} kWrated={kwrated} kWhrated={kwhrated} dispmode=follow daily=StorageCurve_{i+1} {stored}"
        text = shape + curve + "\n" + battery + "\n\n"

        f.write(text)

        linhas.append({
            "Nome": f"Battery_{i + 1}",
            "Tensão (kV)": kv,
            "Bus": bus,
            "kW": kwrated,
            "kWh": kwhrated,
            "Fases": phase
        })

    battery_df = pd.DataFrame(linhas) 

    return battery_df



with open("Battery_30.dss", "w", encoding="utf-8") as f:
    battery_df_30 = code_writer(30, f)

with open("Battery_60.dss", "w", encoding="utf-8") as f:
    battery_df_60 = code_writer(60, f) 





Visualização de Tabelas e Busca de Baterias e Trafos

In [17]:
#display(battery_df_30)

trafo_df = dss_tools.model.transformers_df

def busca_bateria(trafo):
    bus = trafo_df.query(f"name == '{trafo}'")["bus"].iloc[0]
    return battery_df_60.query(f"Bus == '{bus}'")

def busca_trafo(bateria):
    bus = battery_df_60.query(f"Nome == '{bateria}'")["Bus"].iloc[0]
    return trafo_df.query(f"bus == '{bus}'")[["name","bus", "buses", "kv"]]

#Procurar
TRAFO = "trf_5286964a"
BATERIA = "Battery_1"

#Resultado
bateria = busca_bateria(TRAFO)
display(bateria)

trafo = busca_trafo(BATERIA)
display(trafo)


,Nome,Tensão (kV),Bus,kW,kWh,Fases
0,Battery_1,0.38,82036275286964.1.2.3.4,138.286681,553.146725,3
1,Battery_2,0.38,82036275286964.1.2.3.4,138.286681,553.146725,3


,name,bus,buses,kv
127,trf_5286964a,82036275286964.1.2.3.4,"[82036275002086.1.2.3, 82036275286964.1.2.3.4, ]",0.38


In [21]:
lista = [0.32, 0.29, 0.27, 0.27, 0.26, 0.22, 0.01, -0.4, -0.68, -0.85, -0.93, -0.89, -0.73, -0.58, -0.35, -0.11, 0.21, 0.41, 0.49, 0.5, 0.49, 0.46, 0.41, 0.92]
vector = np.array(lista)

print(vector.sum())

0.009999999999999898
